In [1]:
# Install required packages in this notebook (fixes ModuleNotFoundError for sklearn/imblearn)
%pip install scikit-learn imbalanced-learn --quiet

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE

# 1. Load the NEW 50,000-row Dataset
try:
    df = pd.read_csv('dataset.csv')
    print("Dataset 'dataset.csv' loaded successfully.")
    print(f"Total rows: {len(df)}")
except FileNotFoundError:
    print("Error: 'dataset.csv' not found. Make sure the file is in the same directory as this script.")
    exit()

# 2. Prepare the Data (Modified for the new dataset's format)
# Drop irrelevant columns
df = df.drop(['Patient_ID', 'Created_On'], axis=1)

# --- New Data Encoding ---
# Convert text-based 'Yes'/'No' columns to numbers (1/0)
binary_map = {'Yes': 1, 'No': 0}
df['Is_Parent_Carrier'] = df['Is_Parent_Carrier'].map(binary_map)
df['Is_Sibling_Affected'] = df['Is_Sibling_Affected'].map(binary_map)
df['Has_Chronic_Cough'] = df['Has_Chronic_Cough'].map(binary_map)

# Convert 'Newborn_Screen_Flag'
df['Newborn_Screen_Flag'] = df['Newborn_Screen_Flag'].map({'Detected': 1, 'Not_Detected': 0})

# Convert 'Gender' into a numerical format
df = pd.get_dummies(df, columns=['Gender'], drop_first=True)

# Convert the text-based 'Disease_Status' (target) into numbers
# LabelEncoder will automatically assign a number to each class
le = LabelEncoder()
df['Disease_Status'] = le.fit_transform(df['Disease_Status'])

print("\nDisease Status Encoding Complete:")
# Print the mapping so you know which number corresponds to which disease
for i, class_name in enumerate(le.classes_):
    print(f"Class {i}: {class_name}")

# Separate features (X) and target (y)
X = df.drop('Disease_Status', axis=1)
y = df['Disease_Status']

# 3. Split Data into Training and Testing Sets
# We use 'stratify=y' to ensure both train and test sets have a proportional
# representation of all disease classes, which is crucial for imbalanced data.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"\nTraining set size: {X_train.shape[0]} samples (before SMOTE)")
print(f"Testing set size: {X_test.shape[0]} samples")

# 4. Apply SMOTE to the Training Data Only
# This is the best practice to create a powerful and fair model
print("\nApplying SMOTE to balance the training data...")
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)
print(f"Resampled training set shape: {X_train_resampled.shape}")

# 5. Initialize and Train the Random Forest Model
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1) # n_jobs=-1 uses all CPU cores

print("\nTraining the model on the SMOTE-balanced data...")
model.fit(X_train_resampled, y_train_resampled)
print("Model training complete.")

# 6. Make Predictions on the Test Set
print("Making predictions on the original, untouched test data...")
y_pred = model.predict(X_test)

# 7. Evaluate the Model's Performance
accuracy = accuracy_score(y_test, y_pred)
print(f"\nFINAL Model Accuracy: {accuracy * 100:.2f}%")

print("\nFINAL Classification Report:")
# We use 'target_names=le.classes_' to get a readable report with disease names
print(classification_report(y_test, y_pred, target_names=le.classes_))

Note: you may need to restart the kernel to use updated packages.
Dataset 'dataset.csv' loaded successfully.
Total rows: 50000

Disease Status Encoding Complete:
Class 0: Beta_Thalassemia
Class 1: Cystic_Fibrosis
Class 2: Down_Syndrome
Class 3: Healthy
Class 4: Huntingtons_Disease
Class 5: Sickle_Cell_Anemia

Training set size: 40000 samples (before SMOTE)
Testing set size: 10000 samples

Applying SMOTE to balance the training data...
Resampled training set shape: (211296, 13)

Training the model on the SMOTE-balanced data...
Model training complete.
Making predictions on the original, untouched test data...

FINAL Model Accuracy: 84.05%

FINAL Classification Report:
                     precision    recall  f1-score   support

   Beta_Thalassemia       0.68      0.79      0.73       501
    Cystic_Fibrosis       0.07      0.17      0.09        96
      Down_Syndrome       0.03      0.09      0.05       207
            Healthy       0.96      0.89      0.92      8804
Huntingtons_Diseas